# Working with YAML

"[YAML]( https://en.wikipedia.org/wiki/YAML ) is a human-readable data-serialization language. It is commonly used for configuration files and in applications where data is being stored or transmitted."

"Originally YAML was said to mean *Yet Another Markup Language* ... it was then repurposed as *YAML Ain't Markup Language*, a recursive acronym, to distinguish its purpose as data-oriented, rather than document markup."


References:

- [Reading and parsing a YAML file with Python](https://python.land/data-processing/python-yaml)

In [1]:
%%capture
%%bash
apt-get update
apt-get install -y jq tree

## What is YAML?

YAML is a simplified data format for serializing ( i.e. converting to a string ) data structure.  It is a superset of JSON and therefore supports the six main data types/structures of JSON:

- numbers
- strings
- boolean
- nulls
- arrays
- objects/hashes/dictionaries

YAML uses an indented syntax to represent nested objects.  YAML documents start with three dashes `---` on their own line.

Examples:
- number ( unquoted otherwise it will be interpreted as a string ):
```
1234
```
as octal ( becomes 10 decimal )
```
012
```
as hex ( becomes 18 decimal )
```
0x12
```

- string (text can be quoted or unquoted):
```
  Apple
  "Banana"
```
- boolean ( lower case ):
```
true
false
```
- nulls ( lower case ):
```
null
```
- arrays ( surrounded with square brackets ):
```
[ 1 ,2 , 3 ]
```
or each element on its own line prefixed with a dash:

  ```
  - 1
  - 2
  - 3
  ```

- objects:
```
key: value
```
or with curly braces
```
{ key: value }
```
or multi-line with indents
```
key:
    value
```

- nested objects:

  ```
  key:
    value:
    - 1
    - 2
    - 3
  ```

## Why use YAML


It's easier to read ( sometimes ) as it does not have all the curly braces that JSON has.  It is also nicely indented, which goes along with the programming style of Python.  Furthermore, it is often used for configuration, for example, Docker compose files. ( See [ELK Stack example]( https://github.com/docker/awesome-compose/blob/master/elasticsearch-logstash-kibana/compose.yaml ) ).  Lastly, it is easy ( most of the time ) to convert into JSON and back.



In [2]:
!curl -s https://raw.githubusercontent.com/docker/awesome-compose/master/elasticsearch-logstash-kibana/compose.yaml


services:
  elasticsearch:
    image: elasticsearch:7.16.1
    container_name: es
    environment:
      discovery.type: single-node
      ES_JAVA_OPTS: "-Xms512m -Xmx512m"
    ports:
      - "9200:9200"
      - "9300:9300"
    healthcheck:
      test: ["CMD-SHELL", "curl --silent --fail localhost:9200/_cluster/health || exit 1"]
      interval: 10s
      timeout: 10s
      retries: 3
    networks:
      - elastic
  logstash:
    image: logstash:7.16.1
    container_name: log
    environment:
      discovery.seed_hosts: logstash
      LS_JAVA_OPTS: "-Xms512m -Xmx512m"
    volumes:
      - ./logstash/pipeline/logstash-nginx.config:/usr/share/logstash/pipeline/logstash-nginx.config
      - ./logstash/nginx.log:/home/nginx.log
    ports:
      - "5000:5000/tcp"
      - "5000:5000/udp"
      - "5044:5044"
      - "9600:9600"
    depends_on:
      - elasticsearch
    networks:
      - elastic
    command: logstash -f /usr/share/logstash/pipeline/logstash-nginx.config
  kibana:
    image: ki

## Setup

In [3]:
import yaml
import json


In [4]:
%%writefile config.yaml
---
rest:
  url:
    "https://example.org/primenumbers/v1"
  port:
    8443
prime_numbers:
  [2, 3, 5, 7, 11, 13, 17, 19, 23]


Writing config.yaml


In [5]:
! cat -n config.yaml

     1	---
     2	rest:
     3	  url:
     4	    "https://example.org/primenumbers/v1"
     5	  port:
     6	    8443
     7	prime_numbers:
     8	  [2, 3, 5, 7, 11, 13, 17, 19, 23]


In [6]:
ls -l

total 8
-rw-r--r-- 1 root root  126 Nov 16 21:52 config.yaml
drwxr-xr-x 1 root root 4096 Oct 26 22:18 sample_data/


## Reading and parsing (loading) a YAML file


Read YAML into a dictionary.

In [7]:
with open('config.yaml', 'r') as file:
  prime_service = yaml.safe_load(file)
prime_service


{'rest': {'url': 'https://example.org/primenumbers/v1', 'port': 8443},
 'prime_numbers': [2, 3, 5, 7, 11, 13, 17, 19, 23]}

In [8]:
type(prime_service)

dict

In [9]:
prime_service['rest']['url']


'https://example.org/primenumbers/v1'

## Reading and parsing (loading) YAML strings with Python


In [10]:
names_yaml = """
- eric
- justin
- mary-kate
- "10"
"""
names_yaml


'\n- eric\n- justin\n- mary-kate\n- "10"\n'

In [11]:
names = yaml.safe_load(names_yaml)
names


['eric', 'justin', 'mary-kate', '10']

In [12]:
type(names)

list

### Example

Creating a string of YAML text, assign it to a variable, and read the string into a dictionary. Notice the use of triple quotes.

In [13]:
omlet_recipe = '''
Ingredients:
- eggs: 2
- salt: 1 tsp
- water: 1 tbsp

Directions:
- break eggs into a bowl
- wisk in salt and water
- put pan on stove at high heat
- add egg mixture to pan and cook
- put on plate when done
- eat
'''
print(omlet_recipe)



Ingredients:
- eggs: 2
- salt: 1 tsp
- water: 1 tbsp

Directions:
- break eggs into a bowl
- wisk in salt and water
- put pan on stove at high heat
- add egg mixture to pan and cook
- put on plate when done
- eat



In [14]:
omlet_dict = yaml.safe_load(omlet_recipe)
omlet_dict


{'Ingredients': [{'eggs': 2}, {'salt': '1 tsp'}, {'water': '1 tbsp'}],
 'Directions': ['break eggs into a bowl',
  'wisk in salt and water',
  'put pan on stove at high heat',
  'add egg mixture to pan and cook',
  'put on plate when done',
  'eat']}

In [15]:
omlet_dict['Ingredients'][0]["eggs"]


2

## Writing (dumping) YAML to a file



In [16]:
with open('omlet_recipe.yaml', 'w') as file:
  yaml.dump(omlet_dict, file)


In [17]:
!cat -n omlet_recipe.yaml

     1	Directions:
     2	- break eggs into a bowl
     3	- wisk in salt and water
     4	- put pan on stove at high heat
     5	- add egg mixture to pan and cook
     6	- put on plate when done
     7	- eat
     8	Ingredients:
     9	- eggs: 2
    10	- salt: 1 tsp
    11	- water: 1 tbsp


## Convert YAML to JSON

In [18]:
with open('config.yaml', 'r') as file:
  configuration = yaml.safe_load(file)
configuration


{'rest': {'url': 'https://example.org/primenumbers/v1', 'port': 8443},
 'prime_numbers': [2, 3, 5, 7, 11, 13, 17, 19, 23]}

Convert the dictionary to a JSON string.

In [19]:
config_js = json.dumps(configuration)
config_js


'{"rest": {"url": "https://example.org/primenumbers/v1", "port": 8443}, "prime_numbers": [2, 3, 5, 7, 11, 13, 17, 19, 23]}'

Convert the dictionary to a JSON file.

In [20]:
with open('config.json', 'w') as json_file:
    json.dump(configuration, json_file)


In [21]:
!cat -n config.json

     1	{"rest": {"url": "https://example.org/primenumbers/v1", "port": 8443}, "prime_numbers": [2, 3, 5, 7, 11, 13, 17, 19, 23]}

In [22]:
!jq . config.json

{
  "rest": {
    "url": "https://example.org/primenumbers/v1",
    "port": 8443
  },
  "prime_numbers": [
    2,
    3,
    5,
    7,
    11,
    13,
    17,
    19,
    23
  ]
}


Read the JSON file into a dictionary

In [23]:
dict_js = json.dumps(json.load(open('config.json')), indent=2)
print(dict_js)


{
  "rest": {
    "url": "https://example.org/primenumbers/v1",
    "port": 8443
  },
  "prime_numbers": [
    2,
    3,
    5,
    7,
    11,
    13,
    17,
    19,
    23
  ]
}


Does this (the previous cell) open and immediately close the file?

## Convert JSON to YAML

In [24]:
with open('config.json', 'r') as file:
    configuration = json.load(file)
configuration

{'rest': {'url': 'https://example.org/primenumbers/v1', 'port': 8443},
 'prime_numbers': [2, 3, 5, 7, 11, 13, 17, 19, 23]}

In [25]:
with open('config.v02.yaml', 'w') as yaml_file:
    yaml.dump(configuration, yaml_file)


In [26]:
!cat -n config.v02.yaml

     1	prime_numbers:
     2	- 2
     3	- 3
     4	- 5
     5	- 7
     6	- 11
     7	- 13
     8	- 17
     9	- 19
    10	- 23
    11	rest:
    12	  port: 8443
    13	  url: https://example.org/primenumbers/v1


# Converting free text to YAML to JSON

Using the ABQ air quality data.

In [27]:
!curl -s -O https://data.cabq.gov/airquality/aqindex/history/042222.0017


In [28]:
!ls -l 042222.0017


-rw-r--r-- 1 root root 8508 Nov 16 22:11 042222.0017


Convert the first few lines of the air quality data into YAML using `sed` and save to a file.

In [29]:
%%bash
head -30 042222.0017

BEGIN_FILE
FORMAT_VERSION,2
AGENCY,0017
FILENAME,042222.0017
DATA_VERSION,201904222215
TZONE,MST,7
BEGIN_GROUP
VARIABLE,CO
DATA_TYPE,POINT
MEASUREMENT_TYPE,SAMPLE
CHARACTERISTIC,OBSERVED
START_DTG,201904220000
END_DTG,201904222159
INTERVAL,60
START_REF,0
NUMSTEPS,22
AVG_TIME,60
UNITS,PPM
STATIONS,2
BEGIN_DATA
Del Norte HS 1      ,350010023,0.138,0.171,0.196,0.132,0.174,0.272,-999,-999,0.243,0.184,0.12,0.12,0.118,0.125,0.12,0.116,0.118,0.123,0.139,0.123,0.118,0.108
Del Norte HS 1      ,350010023,G,G,G,G,G,G,B,B,G,G,G,G,G,G,G,G,G,G,G,G,G,G
South Valley        ,350010029,0.106,0.059,0.069,0.155,0.209,0.304,0.351,0.307,0.051,0.043,0.045,0.069,0.058,0.127,0.054,0.062,0.048,0.048,0.033,0.023,0.025,0.033
South Valley        ,350010029,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G
END_DATA
END_GROUP
BEGIN_GROUP
VARIABLE,NO2
DATA_TYPE,POINT
MEASUREMENT_TYPE,SAMPLE


In [30]:
!tail -6 042222.0017

BEGIN_DATA
Del Norte HS 2      ,350010023,4.1,0.7,1.9,3,1.8,2.8,1.6,1.3,4,2.3,15.5,14.7,15.1,13.6,14.5,16.9,17.9,7.9,1,10,12.6,12.9
Del Norte HS 2      ,350010023,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G
END_DATA
END_GROUP
END_FILE


In [31]:
%%bash
cat 042222.0017 |
tr -s '\r\n' '\n' |
sed -re '/,/ { s/^/  / }' |
sed -re '/,/! { s/$/:/ }' |
sed -re '{ s/,/: / }' |
sed -re '{ s/: (.*)/: "\1"/ }' |
tee aq.yaml


BEGIN_FILE:
  FORMAT_VERSION: "2"
  AGENCY: "0017"
  FILENAME: "042222.0017"
  DATA_VERSION: "201904222215"
  TZONE: "MST,7"
BEGIN_GROUP:
  VARIABLE: "CO"
  DATA_TYPE: "POINT"
  MEASUREMENT_TYPE: "SAMPLE"
  CHARACTERISTIC: "OBSERVED"
  START_DTG: "201904220000"
  END_DTG: "201904222159"
  INTERVAL: "60"
  START_REF: "0"
  NUMSTEPS: "22"
  AVG_TIME: "60"
  UNITS: "PPM"
  STATIONS: "2"
BEGIN_DATA:
  Del Norte HS 1      : "350010023,0.138,0.171,0.196,0.132,0.174,0.272,-999,-999,0.243,0.184,0.12,0.12,0.118,0.125,0.12,0.116,0.118,0.123,0.139,0.123,0.118,0.108"
  Del Norte HS 1      : "350010023,G,G,G,G,G,G,B,B,G,G,G,G,G,G,G,G,G,G,G,G,G,G"
  South Valley        : "350010029,0.106,0.059,0.069,0.155,0.209,0.304,0.351,0.307,0.051,0.043,0.045,0.069,0.058,0.127,0.054,0.062,0.048,0.048,0.033,0.023,0.025,0.033"
  South Valley        : "350010029,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G"
END_DATA:
END_GROUP:
BEGIN_GROUP:
  VARIABLE: "NO2"
  DATA_TYPE: "POINT"
  MEASUREMENT_TYPE: "SAMPLE"
  CHARAC

In [32]:
!ls -la aq.yaml
!wc aq.yaml

-rw-r--r-- 1 root root 9304 Nov 16 22:13 aq.yaml
 249  584 9304 aq.yaml


In [33]:
!cat aq.yaml


BEGIN_FILE:
  FORMAT_VERSION: "2"
  AGENCY: "0017"
  FILENAME: "042222.0017"
  DATA_VERSION: "201904222215"
  TZONE: "MST,7"
BEGIN_GROUP:
  VARIABLE: "CO"
  DATA_TYPE: "POINT"
  MEASUREMENT_TYPE: "SAMPLE"
  CHARACTERISTIC: "OBSERVED"
  START_DTG: "201904220000"
  END_DTG: "201904222159"
  INTERVAL: "60"
  START_REF: "0"
  NUMSTEPS: "22"
  AVG_TIME: "60"
  UNITS: "PPM"
  STATIONS: "2"
BEGIN_DATA:
  Del Norte HS 1      : "350010023,0.138,0.171,0.196,0.132,0.174,0.272,-999,-999,0.243,0.184,0.12,0.12,0.118,0.125,0.12,0.116,0.118,0.123,0.139,0.123,0.118,0.108"
  Del Norte HS 1      : "350010023,G,G,G,G,G,G,B,B,G,G,G,G,G,G,G,G,G,G,G,G,G,G"
  South Valley        : "350010029,0.106,0.059,0.069,0.155,0.209,0.304,0.351,0.307,0.051,0.043,0.045,0.069,0.058,0.127,0.054,0.062,0.048,0.048,0.033,0.023,0.025,0.033"
  South Valley        : "350010029,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G"
END_DATA:
END_GROUP:
BEGIN_GROUP:
  VARIABLE: "NO2"
  DATA_TYPE: "POINT"
  MEASUREMENT_TYPE: "SAMPLE"
  CHARAC

Read the YAML file into a string variable.

In [34]:
with open('aq.yaml', 'r') as file:
  aq_yaml = file.read()
print(aq_yaml)


BEGIN_FILE:
  FORMAT_VERSION: "2"
  AGENCY: "0017"
  FILENAME: "042222.0017"
  DATA_VERSION: "201904222215"
  TZONE: "MST,7"
BEGIN_GROUP:
  VARIABLE: "CO"
  DATA_TYPE: "POINT"
  MEASUREMENT_TYPE: "SAMPLE"
  CHARACTERISTIC: "OBSERVED"
  START_DTG: "201904220000"
  END_DTG: "201904222159"
  INTERVAL: "60"
  START_REF: "0"
  NUMSTEPS: "22"
  AVG_TIME: "60"
  UNITS: "PPM"
  STATIONS: "2"
BEGIN_DATA:
  Del Norte HS 1      : "350010023,0.138,0.171,0.196,0.132,0.174,0.272,-999,-999,0.243,0.184,0.12,0.12,0.118,0.125,0.12,0.116,0.118,0.123,0.139,0.123,0.118,0.108"
  Del Norte HS 1      : "350010023,G,G,G,G,G,G,B,B,G,G,G,G,G,G,G,G,G,G,G,G,G,G"
  South Valley        : "350010029,0.106,0.059,0.069,0.155,0.209,0.304,0.351,0.307,0.051,0.043,0.045,0.069,0.058,0.127,0.054,0.062,0.048,0.048,0.033,0.023,0.025,0.033"
  South Valley        : "350010029,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G"
END_DATA:
END_GROUP:
BEGIN_GROUP:
  VARIABLE: "NO2"
  DATA_TYPE: "POINT"
  MEASUREMENT_TYPE: "SAMPLE"
  CHARAC

In [35]:
for line in aq_yaml.split("\n"):
  if ": 0" in line:
    print(line)

Convert the string variable into a dictionary.

In [36]:
aq_dict = yaml.safe_load(aq_yaml)
aq_dict


{'BEGIN_FILE': {'FORMAT_VERSION': '2',
  'AGENCY': '0017',
  'FILENAME': '042222.0017',
  'DATA_VERSION': '201904222215',
  'TZONE': 'MST,7'},
 'BEGIN_GROUP': {'VARIABLE': 'WSV',
  'DATA_TYPE': 'POINT',
  'MEASUREMENT_TYPE': 'SAMPLE',
  'CHARACTERISTIC': 'OBSERVED',
  'START_DTG': '201904220000',
  'END_DTG': '201904222159',
  'INTERVAL': '60',
  'START_REF': '0',
  'NUMSTEPS': '22',
  'AVG_TIME': '60',
  'UNITS': 'MPH',
  'STATIONS': '1'},
 'BEGIN_DATA': {'Del Norte HS 2': '350010023,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G'},
 'END_DATA': None,
 'END_GROUP': None,
 'END_FILE': None}

In [37]:
(aq_dict['BEGIN_FILE']['TZONE']).split(",")[0]


'MST'

In [38]:
(aq_dict['BEGIN_GROUP']['INTERVAL']).split(",")[0]


'60'

Convert the dictionary to JSON.

In [39]:
aq_json = json.dumps( aq_dict, indent = 2 )
print(aq_json)


{
  "BEGIN_FILE": {
    "FORMAT_VERSION": "2",
    "AGENCY": "0017",
    "FILENAME": "042222.0017",
    "DATA_VERSION": "201904222215",
    "TZONE": "MST,7"
  },
  "BEGIN_GROUP": {
    "VARIABLE": "WSV",
    "DATA_TYPE": "POINT",
    "MEASUREMENT_TYPE": "SAMPLE",
    "CHARACTERISTIC": "OBSERVED",
    "START_DTG": "201904220000",
    "END_DTG": "201904222159",
    "INTERVAL": "60",
    "START_REF": "0",
    "NUMSTEPS": "22",
    "AVG_TIME": "60",
    "UNITS": "MPH",
    "STATIONS": "1"
  },
  "BEGIN_DATA": {
    "Del Norte HS 2": "350010023,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G,G"
  },
  "END_DATA": null,
  "END_GROUP": null,
  "END_FILE": null
}


### Finding sizes of data files from web page


In [40]:
import pandas as pd
from bs4 import BeautifulSoup as BS4
import requests
import re


In [41]:
url = 'https://data.cabq.gov/airquality/aqindex/history/'
url


'https://data.cabq.gov/airquality/aqindex/history/'

In [42]:
response = requests.get( url )
response


<Response [200]>

In [43]:
dom = BS4( response.text, "html.parser")
type(dom)

bs4.BeautifulSoup

In [45]:
print(pre)

[<pre><img alt="Icon " src="/icons/blank.gif"/> <a href="?C=N;O=D">Name</a>                      <a href="?C=S;O=A">Size</a>  <a href="?C=D;O=A">Description</a><hr/><img alt="[PARENTDIR]" src="/icons/back.gif"/> <a href="/airquality/aqindex/">Parent Directory</a>            -   
<img alt="[   ]" src="/icons/unknown.gif"/> <a href="042222.0017">042222.0017</a>               8.3K  
<img alt="[   ]" src="/icons/unknown.gif"/> <a href="050222.0017">050222.0017</a>                13K  
<img alt="[   ]" src="/icons/unknown.gif"/> <a href="050223.0017">050223.0017</a>                13K  
<img alt="[   ]" src="/icons/unknown.gif"/> <a href="050921.0017">050921.0017</a>                13K  
<img alt="[   ]" src="/icons/unknown.gif"/> <a href="050922.0017">050922.0017</a>                13K  
<img alt="[   ]" src="/icons/unknown.gif"/> <a href="051022.0017">051022.0017</a>               8.2K  
<img alt="[   ]" src="/icons/unknown.gif"/> <a href="051522.0017">051522.0017</a>                13K  

In [44]:
pre = dom.select("pre")
len(pre)


1

In [46]:
lines = str(pre[0]).split("\n")
len(lines)

6413

In [47]:
list(enumerate(lines[:3]))

[(0,
  '<pre><img alt="Icon " src="/icons/blank.gif"/> <a href="?C=N;O=D">Name</a>                      <a href="?C=S;O=A">Size</a>  <a href="?C=D;O=A">Description</a><hr/><img alt="[PARENTDIR]" src="/icons/back.gif"/> <a href="/airquality/aqindex/">Parent Directory</a>            -   '),
 (1,
  '<img alt="[   ]" src="/icons/unknown.gif"/> <a href="042222.0017">042222.0017</a>               8.3K  '),
 (2,
  '<img alt="[   ]" src="/icons/unknown.gif"/> <a href="050222.0017">050222.0017</a>                13K  ')]

In [48]:
list(enumerate(lines[-3:]))

[(0,
  '<img alt="[   ]" src="/icons/unknown.gif"/> <a href="4th%20quarter%20FYFY2016.0017">4th quarter FYFY2016.0017</a> 528K  '),
 (1,
  '<img alt="[   ]" src="/icons/unknown.gif"/> <a href="Pollen.XLS">Pollen.XLS</a>                 19K  '),
 (2, '<hr/></pre>')]

In [49]:
lines.pop()


'<hr/></pre>'

In [50]:
lines[-1]

'<img alt="[   ]" src="/icons/unknown.gif"/> <a href="Pollen.XLS">Pollen.XLS</a>                 19K  '

In [51]:
lines.pop(0)


'<pre><img alt="Icon " src="/icons/blank.gif"/> <a href="?C=N;O=D">Name</a>                      <a href="?C=S;O=A">Size</a>  <a href="?C=D;O=A">Description</a><hr/><img alt="[PARENTDIR]" src="/icons/back.gif"/> <a href="/airquality/aqindex/">Parent Directory</a>            -   '

In [52]:
lines[0]

'<img alt="[   ]" src="/icons/unknown.gif"/> <a href="042222.0017">042222.0017</a>               8.3K  '

In [53]:
len(lines)

6411

In [54]:
filenames =[ BS4(line, "html.parser").find("a").text for line in lines ]
filenames[:10]

['042222.0017',
 '050222.0017',
 '050223.0017',
 '050921.0017',
 '050922.0017',
 '051022.0017',
 '051522.0017',
 '051523.0017',
 '052022.0017',
 '052023.0017']

In [55]:
file_sizes = [ re.split(r' +', BS4(line, "html.parser").text)[-2] for line in lines ]
file_sizes[:10]

['8.3K', '13K', '13K', '13K', '13K', '8.2K', '13K', '13K', '13K', '13K']

In [56]:
df = pd.DataFrame(list(zip(filenames, file_sizes)))
df

,0,1
0,042222.0017,8.3K
1,050222.0017,13K
2,050223.0017,13K
3,050921.0017,13K
4,050922.0017,13K
...,...,...
6406,3rd quarter FY2016.0017,532K
6407,3rd quarter FY2017.0017,531K
6408,4th quarter FY2017.0017,527K
6409,4th quarter FYFY2016.0017,528K


In [57]:
df[1].replace({".*K":"\1*1000",".*M":"\1*1_000_000"}, regex=True)
# .apply(eval)


,1
0,*1000
1,*1000
2,*1000
3,*1000
4,*1000
...,...
6406,*1000
6407,*1000
6408,*1000
6409,*1000


In [58]:
df["sizes"] = (
  df[1].apply(lambda x: float(x.replace("K",""))*1_000 if "K" in x else float(x.replace("M",""))*1_000_000 if "M" in x else x )
)
df



,0,1,sizes
0,042222.0017,8.3K,8300.0
1,050222.0017,13K,13000.0
2,050223.0017,13K,13000.0
3,050921.0017,13K,13000.0
4,050922.0017,13K,13000.0
...,...,...,...
6406,3rd quarter FY2016.0017,532K,532000.0
6407,3rd quarter FY2017.0017,531K,531000.0
6408,4th quarter FY2017.0017,527K,527000.0
6409,4th quarter FYFY2016.0017,528K,528000.0


In [59]:
# an alternative using eval() - usually not recommended
df[1].str.replace("K","*1_000").str.replace("M","*1_000_000").apply(eval).astype(int)


,1
0,8300
1,13000
2,13000
3,13000
4,13000
...,...
6406,532000
6407,531000
6408,527000
6409,528000


In [60]:
df["sizes"].sum()/1_000_000

np.float64(61.0979)

In [61]:
df.columns = ["filename", "file_size_human", "file_size"]
df.head()

,filename,file_size_human,file_size
0,042222.0017,8.3K,8300.0
1,050222.0017,13K,13000.0
2,050223.0017,13K,13000.0
3,050921.0017,13K,13000.0
4,050922.0017,13K,13000.0


In [62]:
df["url"] = "https://data.cabq.gov/airquality/aqindex/history/" + df["filename"]
df

,filename,file_size_human,file_size,url
0,042222.0017,8.3K,8300.0,https://data.cabq.gov/airquality/aqindex/histo...
1,050222.0017,13K,13000.0,https://data.cabq.gov/airquality/aqindex/histo...
2,050223.0017,13K,13000.0,https://data.cabq.gov/airquality/aqindex/histo...
3,050921.0017,13K,13000.0,https://data.cabq.gov/airquality/aqindex/histo...
4,050922.0017,13K,13000.0,https://data.cabq.gov/airquality/aqindex/histo...
...,...,...,...,...
6406,3rd quarter FY2016.0017,532K,532000.0,https://data.cabq.gov/airquality/aqindex/histo...
6407,3rd quarter FY2017.0017,531K,531000.0,https://data.cabq.gov/airquality/aqindex/histo...
6408,4th quarter FY2017.0017,527K,527000.0,https://data.cabq.gov/airquality/aqindex/histo...
6409,4th quarter FYFY2016.0017,528K,528000.0,https://data.cabq.gov/airquality/aqindex/histo...


In [63]:
for url in df["url"][:10]:
  response = requests.get(url)
  print(url, response.text[:20])

https://data.cabq.gov/airquality/aqindex/history/042222.0017 BEGIN_FILE
FORMAT_V
https://data.cabq.gov/airquality/aqindex/history/050222.0017 BEGIN_FILE
FORMAT_V
https://data.cabq.gov/airquality/aqindex/history/050223.0017 BEGIN_FILE
FORMAT_V
https://data.cabq.gov/airquality/aqindex/history/050921.0017 BEGIN_FILE
FORMAT_V
https://data.cabq.gov/airquality/aqindex/history/050922.0017 BEGIN_FILE
FORMAT_V
https://data.cabq.gov/airquality/aqindex/history/051022.0017 BEGIN_FILE
FORMAT_V
https://data.cabq.gov/airquality/aqindex/history/051522.0017 BEGIN_FILE
FORMAT_V
https://data.cabq.gov/airquality/aqindex/history/051523.0017 BEGIN_FILE
FORMAT_V
https://data.cabq.gov/airquality/aqindex/history/052022.0017 BEGIN_FILE
FORMAT_V
https://data.cabq.gov/airquality/aqindex/history/052023.0017 BEGIN_FILE
FORMAT_V
